In [ ]:
from collections.abc import Iterable
import os

import certifi
import urllib3
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEndpoint,
    HuggingFaceEndpointEmbeddings,
 )
from langgraph.graph import MessagesState
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Suppress only the warning emitted when verify=False is used as a fallback.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CA_BUNDLE = certifi.where()

# Requires HUGGINGFACEHUB_API_TOKEN in the notebook environment.
HF_EMBEDDING_MODEL = os.getenv(
    "HF_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
 )
HF_LLM_REPO_ID = os.getenv(
    "HF_LLM_REPO_ID", "mistralai/Mistral-7B-Instruct-v0.3"
 )

List the ECB blog pages that will be downloaded and indexed for question answering.

In [ ]:
urls = [
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260506~586ab63f08.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260421~3c1c5a08ee.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260413~78ef6fe470.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260409~cc951a0d29.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260407~dfa96b8bfc.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260402~d9be74e490.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260331~af8055c801.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260330~0370e4586a.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260327~51b0640c39.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260321~0df0ef78e7.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260312~5dfa697fdd.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260310~b738caf5b4.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260309~3a4841b028.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260304~d9e34fc95f.en.html",
]

Create a loader that retries with different SSL verification settings when a site certificate causes download failures.

In [ ]:
def load_with_ssl_fallback(url: str):
    attempts = [
        ("system-ca", True),
        ("certifi", CA_BUNDLE),
        ("insecure", False),
    ]

    last_error = None
    for mode, verify_value in attempts:
        try:
            loader = WebBaseLoader(
                url,
                requests_kwargs={"verify": verify_value, "timeout": 30},
            )
            if mode == "insecure":
                print(f"Using insecure SSL fallback for {url}")
            return loader.load()
        except Exception as exc:
            last_error = exc
            if mode != "insecure":
                print(f"SSL mode '{mode}' failed for {url}. Trying next fallback.")

    raise RuntimeError(f"All SSL strategies failed for {url}") from last_error

Fetch each page and collect all returned documents into one list for later processing.

In [ ]:
docs = []
for url in urls:
    docs.extend(load_with_ssl_fallback(url))

Flatten nested results so the retriever pipeline always works with standard LangChain Document objects.

In [ ]:
def normalize_documents(items):
    normalized = []
    for item in items:
        if isinstance(item, Document):
            normalized.append(item)
        elif isinstance(item, Iterable) and not isinstance(item, (str, bytes, dict)):
            for inner in item:
                if isinstance(inner, Document):
                    normalized.append(inner)
    return normalized

Break the articles into smaller passages so retrieval works on focused sections instead of full pages.

In [ ]:
docs_list = normalize_documents(docs)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50,
 )
doc_splits = text_splitter.split_documents(docs_list)

Create Hugging Face embeddings, store the chunks in memory, and expose a retriever for semantic search.

In [ ]:
embedding_model = HuggingFaceEndpointEmbeddings(model=HF_EMBEDDING_MODEL)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=embedding_model
 )
retriever = vectorstore.as_retriever()

Keep document lookup in one function so the response step can reuse it cleanly.

In [ ]:
def retrieve_blog_posts(query: str) -> str:
    """Search and return information from ECB blog posts."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

Connect to the Hugging Face inference endpoint that will generate the final natural-language response.

In [ ]:
hf_llm = HuggingFaceEndpoint(
    repo_id=HF_LLM_REPO_ID,
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
 )
response_model = ChatHuggingFace(llm=hf_llm)

Retrieve relevant context for the latest question and ask the model to answer using that evidence.

In [ ]:
def generate_query_or_respond(state: MessagesState):
    """Retrieve relevant ECB posts and answer the latest user question."""
    user_message = state["messages"][-1].content
    context = retrieve_blog_posts(user_message)

    response = response_model.invoke(
        [
            (
                "system",
                "You answer questions about ECB blog posts. Use the retrieved context when it is relevant and say when the context is insufficient.",
            ),
            (
                "human",
                f"Question: {user_message}\n\nRetrieved context:\n{context}",
            ),
        ]
    )
    return {"messages": [response]}

# Exercise: Metadata-Guided Blog Post Analysis with LangChain

In this notebook, you built a retrieval pipeline over ECB blog posts using Hugging Face embeddings and a Hugging Face inference model. The next step is to make the workflow more structured by first inspecting the metadata of each blog post, classifying its subject with an LLM, and then using that classification to decide which follow-up prompts should be run.

The goal of this exercise is to turn the current notebook into a metadata-guided analysis pipeline that combines classification and prompt routing.

### Task:
Use the blog post metadata, such as title, publication date, and URL, as input to an LLM that assigns a subject label to each post.
In particular, you should:
- Extract or organize the available metadata for each blog post.
- Use an LLM to classify each post into a subject category such as inflation, monetary policy, banking, financial stability, digital euro, or another meaningful set of labels.
- Define follow-up prompts that depend on the assigned category.
- Run category-specific prompts after classification, for example a summary prompt for one category and a comparison or risk-analysis prompt for another.
- Check whether the classification is consistent and whether the routed prompts produce more relevant outputs than a single generic prompt.

### Objective:
Build a workflow in which the LLM first uses metadata to understand what a post is about, and then uses that classification to trigger more targeted prompts and more useful downstream analysis.

### Optional bonus (advanced)
- Compare metadata-only classification with classification that also uses retrieved text snippets.
- Allow a post to receive multiple subject labels when appropriate.
- Create a small evaluation table showing the metadata, predicted label, follow-up prompt used, and quality of the final output.

## Solution: Metadata-Guided Classification and Prompt Routing

The cells below implement a complete workflow:

1. Build one metadata record per blog post.
2. Classify each post into a subject label with an LLM (with a robust fallback parser).
3. Route to category-specific follow-up prompts.
4. Compare routed output against a generic prompt.
5. Build a compact evaluation table.

In [1]:
import json
from collections import defaultdict

import pandas as pd
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
def build_post_metadata(docs_list, urls):
    """Create one metadata row per URL using loaded document metadata and inferred title."""
    by_source = defaultdict(list)
    for doc in docs_list:
        source = doc.metadata.get("source") or doc.metadata.get("url")
        if source:
            by_source[source].append(doc)

    rows = []
    for url in urls:
        source_docs = by_source.get(url, [])
        title = ""
        pub_date = ""

        if source_docs:
            md = source_docs[0].metadata
            title = (
                md.get("title")
                or md.get("og:title")
                or md.get("page_title")
                or md.get("dc:title")
                or ""
            )
            pub_date = (
                md.get("publication_date")
                or md.get("published")
                or md.get("date")
                or md.get("article:published_time")
                or ""
            )

            if not title:
                first_nonempty_line = next(
                    (
                        line.strip()
                        for line in source_docs[0].page_content.splitlines()
                        if line.strip()
                    ),
                    "",
                )
                title = first_nonempty_line[:180]

        rows.append(
            {
                "url": url,
                "title": title or "Unknown title",
                "publication_date": pub_date or "Unknown date",
            }
        )

    return pd.DataFrame(rows)


metadata_df = build_post_metadata(docs_list, urls)
metadata_df.head()

In [ ]:
LABELS = [
    "inflation",
    "monetary policy",
    "banking",
    "financial stability",
    "digital euro",
    "payments",
    "labor market",
    "other",
]


def classify_post_with_metadata(row, model=response_model):
    """Classify one post using metadata only. Returns label and short rationale."""
    parser = JsonOutputParser()
    classification_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a careful economic-topic classifier."),
            (
                "human",
                """
You are classifying ECB blog posts from metadata only.

Allowed labels: {labels}.
Return strict JSON with keys: label, rationale.
- label: one of the allowed labels
- rationale: <= 25 words

{format_instructions}

Metadata:
Title: {title}
Publication date: {publication_date}
URL: {url}
""".strip(),
            ),
        ]
    )

    classification_chain = classification_prompt | model | parser
    payload = {
        "labels": ", ".join(LABELS),
        "format_instructions": parser.get_format_instructions(),
        "title": row["title"],
        "publication_date": row["publication_date"],
        "url": row["url"],
    }

    try:
        parsed = classification_chain.invoke(payload)
        label = str(parsed.get("label", "other")).lower().strip()
        if label not in LABELS:
            label = "other"
        rationale = str(parsed.get("rationale", "")).strip()
        raw = json.dumps(parsed)
        return label, rationale, raw
    except Exception:
        fallback_prompt = classification_prompt | model | StrOutputParser()
        raw = fallback_prompt.invoke(payload)
        lowered = raw.lower()
        label = next((x for x in LABELS if x in lowered), "other")
        return label, "Fallback parser used.", raw


classified_rows = []
for _, row in metadata_df.iterrows():
    label, rationale, raw_response = classify_post_with_metadata(row)
    classified_rows.append(
        {
            **row.to_dict(),
            "predicted_label": label,
            "classification_rationale": rationale,
            "raw_classification_response": raw_response,
        }
    )

classified_df = pd.DataFrame(classified_rows)
classified_df[["title", "publication_date", "predicted_label"]].head()

In [ ]:
CATEGORY_PROMPTS = {
    "inflation": "Summarize the inflation-related argument, main drivers, and policy implications in 4 bullet points.",
    "monetary policy": "Extract the policy stance, transmission channels, and expected macro effects in 4 bullet points.",
    "banking": "Identify banking-sector issues, supervisory considerations, and balance-sheet implications in 4 bullet points.",
    "financial stability": "Provide a risk-focused analysis: vulnerabilities, triggers, and mitigating policies in 4 bullet points.",
    "digital euro": "Summarize goals, design trade-offs, and implementation challenges for the digital euro in 4 bullet points.",
    "payments": "Summarize payment-system topics, frictions, and proposed solutions in 4 bullet points.",
    "labor market": "Summarize labor-market dynamics, wage mechanisms, and implications for inflation/policy in 4 bullet points.",
    "other": "Provide a concise analytical summary with key message, evidence, and policy relevance in 4 bullet points.",
}


def answer_with_category_prompt(row, model=response_model):
    category = row["predicted_label"]
    routed_instruction = CATEGORY_PROMPTS.get(category, CATEGORY_PROMPTS["other"])

    context = retrieve_blog_posts(row["title"])

    routed_chain = (
        ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You answer questions about ECB blog posts. Use retrieved context when relevant and say when context is insufficient.",
                ),
                (
                    "human",
                    """
Post metadata:
- Title: {title}
- Publication date: {publication_date}
- URL: {url}
- Predicted category: {category}

Task:
{task}

Retrieved context:
{context}
""".strip(),
                ),
            ]
        )
        | model
        | StrOutputParser()
    )

    generic_chain = (
        ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You answer questions about ECB blog posts. Use retrieved context when relevant and say when context is insufficient.",
                ),
                (
                    "human",
                    """
Post metadata:
- Title: {title}
- Publication date: {publication_date}
- URL: {url}

Task:
Provide a general summary of this post in 4 bullet points.

Retrieved context:
{context}
""".strip(),
                ),
            ]
        )
        | model
        | StrOutputParser()
    )

    payload = {
        "title": row["title"],
        "publication_date": row["publication_date"],
        "url": row["url"],
        "category": category,
        "task": routed_instruction,
        "context": context,
    }

    routed_answer = routed_chain.invoke(payload)
    generic_answer = generic_chain.invoke(payload)

    routed_prompt = f"Category: {category}. Task: {routed_instruction}"
    generic_prompt = "Provide a general summary of this post in 4 bullet points."

    return routed_prompt, routed_answer, generic_prompt, generic_answer


analysis_rows = []
for _, row in classified_df.iterrows():
    routed_prompt, routed_answer, generic_prompt, generic_answer = answer_with_category_prompt(row)
    analysis_rows.append(
        {
            **row.to_dict(),
            "follow_up_prompt": routed_prompt,
            "routed_output": routed_answer,
            "generic_prompt": generic_prompt,
            "generic_output": generic_answer,
        }
    )

analysis_df = pd.DataFrame(analysis_rows)
analysis_df[["title", "predicted_label", "url"]].head()

In [ ]:
def quick_quality_score(text: str) -> int:
    """Very lightweight proxy score for demonstration (1-5)."""
    if not text or len(text.strip()) < 40:
        return 1
    score = 2
    if "-" in text or "\n" in text:
        score += 1
    if any(k in text.lower() for k in ["risk", "policy", "inflation", "financial", "bank"]):
        score += 1
    if "insufficient" in text.lower() or "not enough" in text.lower():
        score += 1
    return min(score, 5)


analysis_df["routed_quality_score"] = analysis_df["routed_output"].map(quick_quality_score)
analysis_df["generic_quality_score"] = analysis_df["generic_output"].map(quick_quality_score)
analysis_df["score_delta_routed_minus_generic"] = (
    analysis_df["routed_quality_score"] - analysis_df["generic_quality_score"]
)

final_columns = [
    "title",
    "publication_date",
    "url",
    "predicted_label",
    "classification_rationale",
    "follow_up_prompt",
    "routed_quality_score",
    "generic_quality_score",
    "score_delta_routed_minus_generic",
]

evaluation_df = analysis_df[final_columns].copy()
evaluation_df

In [ ]:
# Optional bonus: allow multi-label prediction using metadata + short retrieved snippet.
def classify_post_multilabel(row, model=response_model):
    snippet = retrieve_blog_posts(row["title"])[:1200]
    parser = JsonOutputParser()

    multilabel_chain = (
        ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You classify ECB blog posts into possibly multiple categories.",
                ),
                (
                    "human",
                    """
Assign 1-3 labels from: {labels}.
Return strict JSON with keys: labels, rationale.

{format_instructions}

Metadata:
Title: {title}
Publication date: {publication_date}
URL: {url}

Snippet:
{snippet}
""".strip(),
                ),
            ]
        )
        | model
        | parser
    )

    payload = {
        "labels": ", ".join(LABELS),
        "format_instructions": parser.get_format_instructions(),
        "title": row["title"],
        "publication_date": row["publication_date"],
        "url": row["url"],
        "snippet": snippet,
    }

    try:
        parsed = multilabel_chain.invoke(payload)
        labels = [str(x).lower().strip() for x in parsed.get("labels", [])]
        labels = [x for x in labels if x in LABELS]
        labels = labels[:3] if labels else ["other"]
        return labels, str(parsed.get("rationale", "")).strip()
    except Exception:
        return ["other"], "Fallback parser used."


bonus_df = metadata_df.copy()
bonus_df[["multilabels", "multilabel_rationale"]] = bonus_df.apply(
    lambda r: pd.Series(classify_post_multilabel(r)), axis=1
)
bonus_df[["title", "multilabels"]].head()